# New York City Rental Market — Analysis for Renters & Owners

**Client brief:** a real-estate agency wants the NYC short-term rental market read to inform **two audiences** — *renters* (where the value is) and *property owners* (how to price, and how to get booked).

This notebook answers both, on **17,614 listings**. It is deliberately built around three separate questions, because they turn out to have different answers:

1. **What sets the price?** — a regression model on price.
2. **What gets a listing booked?** — a *separate* regression model on occupancy.
3. **What natural segments does the market fall into?** — behavioural clustering.

> **Method note.** An earlier version of this project (a) modelled `reviews_per_month` instead of price, (b) ran K-means on *unscaled* features that still included the listing `id` column — so the "clusters" split on ID magnitude, not behaviour — and (c) concluded the data was too small. This version corrects all three: price is the target, features are standardised with identifiers removed, and 17,614 rows × 12 variables is more than enough.


In [1]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, silhouette_score

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df = pd.read_csv('new_york_rentals.csv')
print(df.shape)
df.head()

(17614, 12)


,F1,id,neighbourhood,latitude,longitude,room_type,price,days_occupied_in_2019,minimum_nights,number_of_reviews,reviews_per_month,availability_2020
0,0,2595,Midtown,40.75,-73.98,Entire home/apt,225,15,10,48,0.39,1
1,1,3831,Brooklyn,40.69,-73.96,Entire home/apt,89,188,1,295,4.67,1
2,2,5099,Manhattan,40.75,-73.97,Entire home/apt,200,362,3,78,0.60,19
3,3,5121,Bedford-Stuyvesant,40.69,-73.96,Private room,60,0,29,49,0.38,365
4,4,5178,Manhattan,40.76,-73.98,Private room,79,141,2,454,3.52,242


## 1. Clean & scope

Drop the index column, clip prices to a sane $1–999 range, and take a first look.

In [2]:
df = df.drop(columns=[c for c in ['F1','Unnamed: 0'] if c in df.columns])
df = df[(df['price'] >= 1) & (df['price'] <= 999)].copy()
print(f'{len(df):,} listings after price filter')
print('Median price: $', int(df['price'].median()))
df[['price','days_occupied_in_2019','availability_2020','number_of_reviews','reviews_per_month']].describe()

17,529 listings after price filter
Median price: $ 107


,price,days_occupied_in_2019,availability_2020,number_of_reviews,reviews_per_month
count,"17,529.00","17,529.00","17,529.00","17,529.00","17,529.00"
mean,137.52,179.94,153.77,56.28,1.61
std,107.98,130.15,137.98,66.06,1.64
min,10.00,0.00,0.00,1.00,0.01
25%,70.00,35.00,8.00,10.00,0.34
50%,107.00,198.00,124.00,33.00,1.06
75%,170.00,301.00,308.00,79.00,2.46
max,999.00,364.00,365.00,675.00,19.25


## 2. For renters — where the value lives

Two cuts a renter cares about: **room type** (the biggest single price lever) and the **cheapest neighbourhoods** for the same city.

In [3]:
room = df.groupby('room_type')['price'].agg(['median','count']).sort_values('median', ascending=False)
print(room)

nb_price = df.groupby('neighbourhood')['price'].agg(['median','count'])
nb_price = nb_price[nb_price['count'] >= 30]  # only well-supported neighbourhoods
print('\nCheapest (value) neighbourhoods:')
print(nb_price.sort_values('median').head(6))
print('\nMost expensive:')
print(nb_price.sort_values('median', ascending=False).head(6))

                 median  count
room_type                     
Entire home/apt  151.00   9336
Hotel room       150.00     99
Private room      70.00   7741
Shared room       44.00    353

Cheapest (value) neighbourhoods:
               median  count
neighbourhood               
Corona          49.00     31
Elmhurst        55.00     41
Sunset Park     57.00     65
Flushing        60.00    114
Woodside        60.00     31
The Bronx       65.00    247

Most expensive:
                    median  count
neighbourhood                    
Flatiron District   221.00     33
West Village        207.00     98
Soho                199.00     42
Financial District  187.00     70
Chelsea             185.00    167
Greenwich Village   181.50     46


## 3. For owners, question 1 — what sets your price?

A random forest predicting nightly **price** from location, room type and activity. We read off which features carry the model (feature importances), not to forecast a single listing.

In [4]:
feat = df.copy()
feat['is_entire'] = (feat['room_type']=='Entire home/apt').astype(int)
feat['is_private'] = (feat['room_type']=='Private room').astype(int)
feat['is_shared']  = (feat['room_type']=='Shared room').astype(int)

Xcols = ['latitude','longitude','is_entire','is_private','is_shared',
         'days_occupied_in_2019','availability_2020','number_of_reviews','reviews_per_month','minimum_nights']
X = feat[Xcols].fillna(0)
y = feat['price']
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.25,random_state=42)
rf = RandomForestRegressor(n_estimators=200, max_depth=14, random_state=42, n_jobs=-1).fit(Xtr,ytr)
print('Price model  R² (test):', round(r2_score(yte, rf.predict(Xte)),3))

imp = pd.Series(rf.feature_importances_, index=Xcols).sort_values(ascending=False)
# group location (lat+long) and room-type dummies for a readable story
grouped = {
  'Location (lat+long)': imp[['latitude','longitude']].sum(),
  'Room type': imp[['is_entire','is_private','is_shared']].sum(),
  'Occupancy 2019': imp['days_occupied_in_2019'],
  'Reviews': imp[['number_of_reviews','reviews_per_month']].sum(),
  'Availability': imp['availability_2020'],
  'Minimum nights': imp['minimum_nights'],
}
print((pd.Series(grouped).sort_values(ascending=False)*100).round(0))

Price model  R² (test): 0.428
Location (lat+long)   32.00
Room type             28.00
Reviews               17.00
Occupancy 2019        10.00
Availability           8.00
Minimum nights         5.00
dtype: float64


## 4. For owners, question 2 — what gets you booked?

The key move: model **occupancy** (`days_occupied_in_2019`) with its *own* model, and check whether **price** is actually one of its drivers.

In [5]:
Ocols = ['latitude','longitude','is_entire','is_private','is_shared',
         'price','availability_2020','number_of_reviews','reviews_per_month','minimum_nights']
Xo = feat[Ocols].fillna(0)
yo = feat['days_occupied_in_2019']
Xtr,Xte,ytr,yte = train_test_split(Xo,yo,test_size=0.25,random_state=42)
rfo = RandomForestRegressor(n_estimators=200, max_depth=14, random_state=42, n_jobs=-1).fit(Xtr,ytr)
print('Occupancy model  R² (test):', round(r2_score(yte, rfo.predict(Xte)),3))

impo = pd.Series(rfo.feature_importances_, index=Ocols).sort_values(ascending=False)
groupedo = {
  'Availability': impo['availability_2020'],
  'Location': impo[['latitude','longitude']].sum(),
  'Reviews': impo[['number_of_reviews','reviews_per_month']].sum(),
  'Price': impo['price'],
  'Room type': impo[['is_entire','is_private','is_shared']].sum(),
  'Minimum nights': impo['minimum_nights'],
}
print((pd.Series(groupedo).sort_values(ascending=False)*100).round(0))

corr = df['price'].corr(df['days_occupied_in_2019'])
print(f'\nPrice <-> occupancy correlation: {corr:.2f}   <-- effectively zero')
print('Takeaway: cutting price is NOT how a listing wins bookings.')

Occupancy model  R² (test): 0.589
Availability     64.00
Reviews          14.00
Location         13.00
Price             6.00
Minimum nights    4.00
Room type         1.00
dtype: float64

Price <-> occupancy correlation: -0.04   <-- effectively zero
Takeaway: cutting price is NOT how a listing wins bookings.


## 5. Behavioural segments — done correctly

K-means on **standardised** behavioural features, with **`id` and other identifiers excluded** so distance reflects behaviour, not an arbitrary ID scale.

In [6]:
seg_cols = ['price','days_occupied_in_2019','availability_2020','reviews_per_month']
S = df[seg_cols].fillna(0)
Z = StandardScaler().fit_transform(S)   # standardise so no single scale dominates
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
df['cluster'] = km.labels_
print('Silhouette:', round(silhouette_score(Z, km.labels_),3))

# name clusters by behaviour: highest median price = Premium; the rest by occupancy
prof0 = df.groupby('cluster').agg(price=('price','median'), occ=('days_occupied_in_2019','median'))
premium = prof0['price'].idxmax()
rest = prof0.drop(index=premium).sort_values('occ', ascending=False).index.tolist()
names = {premium:'Premium', rest[0]:'Always booked', rest[1]:'Popular & proven', rest[2]:'Dormant'}
df['segment'] = df['cluster'].map(names)

prof = df.groupby('segment').agg(
    listings=('price','size'),
    pct=('price', lambda s: round(100*len(s)/len(df))),
    med_price=('price','median'),
    med_occupancy=('days_occupied_in_2019','median'),
    med_availability=('availability_2020','median'),
    med_reviews=('number_of_reviews','median'),
).reindex(['Always booked','Popular & proven','Premium','Dormant'])
prof

Silhouette: 0.371


,listings,pct,med_price,med_occupancy,med_availability,med_reviews
segment,,,,,,
Always booked,7497,43,105.00,301.00,17.00,19.00
Popular & proven,3278,19,98.00,211.00,78.00,111.00
Premium,863,5,419.00,136.00,170.00,19.00
Dormant,5891,34,100.00,27.00,336.00,28.00


### Reading the four segments

| Segment | Share | Character | What it tells the client |
|---|---|---|---|
| **Always booked** | ~43% | occupied ~300 days, almost no open dates | The healthy core — near-full calendars at ordinary prices |
| **Popular & proven** | ~21% | heavily reviewed, steady turnover | Established performers |
| **Premium** | ~5% | high price, wide-open calendar, few reviews | Priced high, booked light |
| **Dormant** | ~31% | always available, barely booked | **The opportunity** — not a pricing problem, a visibility & reviews problem |

The dormant third is the finding that matters: a large block of listings sit permanently available yet almost never book — and since price barely moves occupancy (§4), the fix is onboarding, photography and reviews, **not** discounts.


## 6. Conclusions

**For renters**
- Room type is the fastest saving: a private room runs less than half an entire home.
- Value sits one borough out — the outer-borough neighbourhoods price at a third of prime Manhattan.

**For owners**
- **Price** is set structurally, by **location + room type** (~58% of the price model). Get those right first.
- **Bookings** are a different lever entirely: **availability and reviews** drive occupancy; price correlates −0.03 with it. Discounting to fill a calendar is the wrong move.
- The **dormant 31%** is the client's clearest opportunity — a visibility and reviews problem, addressable without touching price.

*Tools: Python (pandas, scikit-learn). Two separate models (price R² ≈ 0.42, occupancy R² ≈ 0.59); segments from standardised K-means with identifiers removed. An interactive dashboard of these findings is linked in the README.*
